# **1. Setup & Libraries**

In [ ]:
!pip install matplotlib-scalebar
!pip install contextily

# For post-hoc seasonal tests
!pip install scikit-posthocs

!pip install pymannkendall


In [ ]:
import os
import glob
import calendar

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.ticker import ScalarFormatter
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn as sns

from scipy.stats import mannwhitneyu, kruskal, wilcoxon
import pymannkendall as mk
import scikit_posthocs as sp


# **2. Load Dataset**

In [ ]:
file_path = r"C:\Users\pc\Documents\Rainfall\Data\Final processed 3-hourly rainfall data.csv"
df_rain = pd.read_csv(file_path)
df_rain

,StationID,StationName,Latitude,Longitude,Primary_Region,Secondary_Region,Datetime,Year,Month,Day,Time,Season,DewPointTemperature,StationLevelPressure,SP,DR,Humidity,Rainfall
0,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 00:00:00,2003,1,1,0,Winter,12.0,1009.7,0.0,0.0,91.0,0.0
1,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 03:00:00,2003,1,1,3,Winter,13.0,1011.3,0.0,0.0,74.0,0.0
2,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 06:00:00,2003,1,1,6,Winter,15.0,1011.2,4.0,31.0,57.0,0.0
3,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 09:00:00,2003,1,1,9,Winter,9.0,1010.3,2.0,5.0,35.0,0.0
4,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 12:00:00,2003,1,1,12,Winter,13.0,1010.4,0.0,0.0,61.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2116614,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 09:00:00,2023,7,31,9,Monsoon,26.0,1016.0,1.0,10.0,73.0,0.0
2116615,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 12:00:00,2023,7,31,12,Monsoon,26.0,1016.0,0.0,0.0,82.0,0.0
2116616,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 15:00:00,2023,7,31,15,Monsoon,26.0,1016.0,0.0,0.0,88.0,0.0
2116617,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 18:00:00,2023,7,31,18,Monsoon,26.0,1016.0,0.0,0.0,88.0,0.0


In [ ]:
# REMOVE THE LATE STATION (Example ID: 10325)
# Replace 10325 with the actual ID of the 2008 station
stations_to_drop = [11921]
df_rain = df_rain[~df_rain["StationID"].isin(stations_to_drop)].reset_index(drop=True)
df_rain


,StationID,StationName,Latitude,Longitude,Primary_Region,Secondary_Region,Datetime,Year,Month,Day,Time,Season,DewPointTemperature,StationLevelPressure,SP,DR,Humidity,Rainfall
0,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 00:00:00,2003,1,1,0,Winter,12.0,1009.7,0.0,0.0,91.0,0.0
1,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 03:00:00,2003,1,1,3,Winter,13.0,1011.3,0.0,0.0,74.0,0.0
2,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 06:00:00,2003,1,1,6,Winter,15.0,1011.2,4.0,31.0,57.0,0.0
3,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 09:00:00,2003,1,1,9,Winter,9.0,1010.3,2.0,5.0,35.0,0.0
4,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 12:00:00,2003,1,1,12,Winter,13.0,1010.4,0.0,0.0,61.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2069862,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 09:00:00,2023,7,31,9,Monsoon,26.0,1016.0,1.0,10.0,73.0,0.0
2069863,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 12:00:00,2023,7,31,12,Monsoon,26.0,1016.0,0.0,0.0,82.0,0.0
2069864,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 15:00:00,2023,7,31,15,Monsoon,26.0,1016.0,0.0,0.0,88.0,0.0
2069865,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 18:00:00,2023,7,31,18,Monsoon,26.0,1016.0,0.0,0.0,88.0,0.0


# **Persistent**

In [ ]:
# ============================================================
# SCRIPT A: PERSISTENCE BASELINE
# ============================================================

import os, random, math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score

# ------------------------------------------------------------
# 0) Reproducibility + Device
# ------------------------------------------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ------------------------------------------------------------
# 0.5) Paths (EDIT)
# ------------------------------------------------------------
RESULTS_SAVE_DIR = r"C:\Users\pc\Documents\Rainfall\Baselines\Persistence\v1"
os.makedirs(RESULTS_SAVE_DIR, exist_ok=True)
print("Results directory:", RESULTS_SAVE_DIR)

# ============================================================
# 1) Utilities (same style as other baselines)
# ============================================================

class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.nanstd(X, axis=0)
        self.std_  = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_


def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()


def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    losses = []
    for k, q in enumerate(quantiles):
        losses.append(pinball_loss(y_hat[..., k], y_true, q, mask=mask))
    return 2.0 * torch.stack(losses).mean()


def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()


def safe_div(a, b, eps=1e-12):
    return a / (b + eps)


def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {
            "n_valid": int(valid.sum()),
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds},
        }

    p = probs[valid]
    y = y_true[valid]

    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try:
        roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except Exception:
        roc_auc = np.nan

    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP = ((yhat == 1) & (y == 1)).sum()
        FP = ((yhat == 1) & (y == 0)).sum()
        FN = ((yhat == 0) & (y == 1)).sum()

        POD = safe_div(TP, TP + FN)
        FAR = safe_div(FP, TP + FP)
        CSI = safe_div(TP, TP + FP + FN)

        by_thr[thr] = {"POD": float(POD), "FAR": float(FAR), "CSI": float(CSI)}

    return {
        "n_valid": int(valid.sum()),
        "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan,
        "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan,
        "by_thr": by_thr,
    }


def make_splits(T, train_frac=0.7, val_frac=0.15):
    train_end = int(T * train_frac)
    val_end   = int(T * (train_frac + val_frac))
    return train_end, val_end


def scale_inputs_train_only(X_raw, train_end):
    T, N, F = X_raw.shape
    X_flat = X_raw.reshape(T*N, F)
    train_flat = X_flat[:train_end*N]
    scaler = NaNIgnoringStandardScaler().fit(train_flat)
    X_scaled = scaler.transform(X_flat).reshape(T, N, F).astype(np.float32)
    return X_scaled, scaler


def compute_thresholds_train(Y, M, train_end, q=0.95):
    Y_train = Y[:train_end]
    M_train = M[:train_end]
    N = Y.shape[1]
    thr = np.zeros(N, dtype=np.float32)

    global_vals = Y_train[M_train > 0.5]
    global_fallback = np.nanpercentile(global_vals, q*100) if global_vals.size > 0 else 0.0

    for j in range(N):
        vals = Y_train[:, j][M_train[:, j] > 0.5]
        if len(vals) < 100:
            thr[j] = global_fallback
        else:
            thr[j] = np.nanpercentile(vals, q*100)
    return thr


def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc  = np.full((T, N), np.nan, dtype=np.float32)
    Mask = np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window = Y_rain[t+1:t+1+H_out]
        wmask  = M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok] = 1.0
        Acc[t, ok]  = window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing (same as before)
# ============================================================

def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)

    dr = df["DR"].astype(np.float32).to_numpy()
    dr_rad = np.deg2rad(dr)
    df["DR_sin"] = np.sin(dr_rad)
    df["DR_cos"] = np.cos(dr_rad)

    stations = (
        df[["StationID", "StationName", "Latitude", "Longitude"]]
        .drop_duplicates()
        .sort_values("StationID")
        .reset_index(drop=True)
    )
    stations["node_index"] = np.arange(len(stations))
    N = len(stations)
    print("Total stations:", N)

    times = np.sort(df["Datetime"].unique())
    T = len(times)
    print("Total unique times:", T)

    full_index = pd.MultiIndex.from_product(
        [times, stations["StationID"].values],
        names=["Datetime", "StationID"]
    )

    meteo_cols = [
        "DewPointTemperature",
        "StationLevelPressure",
        "SP",
        "Humidity",
        "Rainfall",
        "DR_sin",
        "DR_cos",
    ]

    df2 = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index()
    df_full = df2.reindex(full_index)
    X_raw = df_full.values.reshape(T, N, len(meteo_cols)).astype(np.float32)

    M_in = (~np.isnan(X_raw)).astype(np.float32)

    rain_idx = meteo_cols.index("Rainfall")
    Y_rain = X_raw[..., rain_idx]
    M_y    = M_in[..., rain_idx]

    dt_index = pd.DatetimeIndex(times)
    hour = dt_index.hour.astype(np.float32)
    month = dt_index.month.astype(np.float32)

    hour_sin = np.sin(2*np.pi*(hour/24.0))
    hour_cos = np.cos(2*np.pi*(hour/24.0))
    month_sin = np.sin(2*np.pi*((month-1)/12.0))
    month_cos = np.cos(2*np.pi*((month-1)/12.0))

    time_feats = np.stack([hour_sin, hour_cos, month_sin, month_cos], axis=-1).astype(np.float32)
    time_feats = np.repeat(time_feats[:, None, :], N, axis=1)

    season_by_time = (
        df.groupby("Datetime")["Season"]
          .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
          .reindex(times)
          .astype(str)
          .values
    )
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}
    season_ids = np.array([season_to_id[s] for s in season_by_time], dtype=np.int64)

    return {
        "stations": stations,
        "times": times,
        "X_raw": X_raw,
        "M_in": M_in,
        "Y_rain": Y_rain,
        "M_y": M_y,
        "meteo_cols": meteo_cols,
        "time_feats": time_feats,
        "season_ids": season_ids,
        "unique_seasons": unique_seasons,
    }

# ============================================================
# 3) Dataset
# ============================================================

class BanglaRainDataset(Dataset):
    def __init__(
        self,
        X_scaled, M_in, time_feats,
        Y_rain, M_y,
        Acc24, MaskAcc24,
        season_ids,
        thr3h, thrAcc24,
        t_start, t_end,
        T_in=16, H_out=8,
        peak_min_obs=None,
    ):
        self.X_scaled = X_scaled
        self.M_in = M_in
        self.time_feats = time_feats
        self.Y_rain = Y_rain
        self.M_y = M_y
        self.Acc24 = Acc24
        self.MaskAcc24 = MaskAcc24
        self.season_ids = season_ids
        self.thr3h = thr3h
        self.thrAcc24 = thrAcc24
        self.T_in = T_in
        self.H_out = H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)

        self.indices = []
        for t in range(t_start + (T_in - 1), t_end - H_out):
            self.indices.append(t)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]

        x  = self.X_scaled[t-self.T_in+1:t+1]     # [T_in,N,F]
        m  = self.M_in[t-self.T_in+1:t+1]         # [T_in,N,F]
        tf = self.time_feats[t-self.T_in+1:t+1]   # [T_in,N,4]

        x = np.nan_to_num(x, nan=0.0).astype(np.float32)
        x_all = np.concatenate([x, m.astype(np.float32), tf], axis=-1)  # [T_in,N,F+F+4]

        y  = self.Y_rain[t+1:t+1+self.H_out]       # [H_out,N]
        my = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y, nan=0.0)).astype(np.float32)

        # flash at t+1
        y_next = self.Y_rain[t+1]
        m_next = self.M_y[t+1].astype(np.float32)
        flash  = ((y_next >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mflash = m_next.copy()

        # peak24
        y_win = self.Y_rain[t+1:t+1+self.H_out]
        m_win = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)
        y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)

        # acc24
        acc  = self.Acc24[t]
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((acc >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)

        regime_id = int(self.season_ids[t])

        return (
            torch.from_numpy(x_all),                         # x
            torch.tensor(regime_id, dtype=torch.long),       # regime
            torch.from_numpy(y_log),                         # y_log
            torch.from_numpy(my),                            # my
            torch.from_numpy(flash), torch.from_numpy(mflash),
            torch.from_numpy(peak24), torch.from_numpy(mpeak),
            torch.from_numpy(acc24),  torch.from_numpy(macc),
        )

# ============================================================
# 4) Evaluation (same as other baselines)
# ============================================================

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.5)):
    model.eval()

    total_crps_log = 0.0
    total_crps_mm  = 0.0
    total_brier_flash = 0.0
    total_brier_peak  = 0.0
    total_brier_acc   = 0.0
    nb = 0

    H_out = None
    sum_abs = None
    sum_sq  = None
    count   = None

    qs = np.array(list(quantiles), dtype=np.float32)
    k50 = int(np.argmin(np.abs(qs - 0.5)))

    flash_p_list, flash_y_list, flash_m_list = [], [], []
    peak_p_list,  peak_y_list,  peak_m_list  = [], [], []
    acc_p_list,   acc_y_list,   acc_m_list   = [], [], []

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in loader:
        x = x.to(device)
        reg = reg.to(device)
        y_log = y_log.to(device)
        my = my.to(device)
        flash = flash.to(device); mflash = mflash.to(device)
        peak  = peak.to(device);  mpeak  = mpeak.to(device)
        acc   = acc.to(device);   macc   = macc.to(device)

        q_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        total_crps_log += crps_from_quantiles(q_hat, y_log, quantiles, mask=my).item()

        q_mm = torch.expm1(q_hat).clamp_min(0.0)
        y_mm = torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm  += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        p_flash = torch.sigmoid(flash_logits)
        p_peak  = torch.sigmoid(peak_logits)
        p_acc   = torch.sigmoid(acc_logits)

        total_brier_flash += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak  += brier_score(p_peak,  peak,  mask=mpeak ).item()
        total_brier_acc   += brier_score(p_acc,   acc,   mask=macc  ).item()

        nb += 1

        q50_log = q_hat[..., k50]
        q50_mm  = torch.expm1(q50_log).clamp_min(0.0)

        B, H, N = y_mm.shape
        if H_out is None:
            H_out = H
            sum_abs = torch.zeros(H_out, device="cpu")
            sum_sq  = torch.zeros(H_out, device="cpu")
            count   = torch.zeros(H_out, device="cpu")

        for h in range(H):
            m = (my[:, h, :] > 0.5)
            if m.any():
                err = (q50_mm[:, h, :][m] - y_mm[:, h, :][m]).detach().cpu()
                sum_abs[h] += err.abs().sum()
                sum_sq[h]  += (err**2).sum()
                count[h]   += m.sum().detach().cpu()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1))
        flash_y_list.append(flash.detach().cpu().reshape(-1))
        flash_m_list.append(mflash.detach().cpu().reshape(-1))

        peak_p_list.append(p_peak.detach().cpu().reshape(-1))
        peak_y_list.append(peak.detach().cpu().reshape(-1))
        peak_m_list.append(mpeak.detach().cpu().reshape(-1))

        acc_p_list.append(p_acc.detach().cpu().reshape(-1))
        acc_y_list.append(acc.detach().cpu().reshape(-1))
        acc_m_list.append(macc.detach().cpu().reshape(-1))

    out = {
        "CRPS_log":   total_crps_log / max(nb, 1),
        "CRPS_mm":    total_crps_mm  / max(nb, 1),
        "Brier_flash": total_brier_flash / max(nb, 1),
        "Brier_peak":  total_brier_peak  / max(nb, 1),
        "Brier_acc":   total_brier_acc   / max(nb, 1),
    }

    maes, rmses, counts = [], [], []
    for h in range(H_out):
        if count[h].item() < 1:
            maes.append(np.nan); rmses.append(np.nan); counts.append(0)
        else:
            maes.append(float((sum_abs[h] / count[h]).item()))
            rmses.append(float(torch.sqrt(sum_sq[h] / count[h]).item()))
            counts.append(int(count[h].item()))
    out.update({"MAE_mm_by_lead": maes, "RMSE_mm_by_lead": rmses, "n_valid_by_lead": counts})

    flash_p = torch.cat(flash_p_list).numpy()
    flash_y = torch.cat(flash_y_list).numpy()
    flash_m = torch.cat(flash_m_list).numpy()

    peak_p  = torch.cat(peak_p_list).numpy()
    peak_y  = torch.cat(peak_y_list).numpy()
    peak_m  = torch.cat(peak_m_list).numpy()

    acc_p   = torch.cat(acc_p_list).numpy()
    acc_y   = torch.cat(acc_y_list).numpy()
    acc_m   = torch.cat(acc_m_list).numpy()

    out["Flash_metrics"]  = event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds)
    out["Peak24_metrics"] = event_metrics_binary(peak_p,  peak_y,  peak_m,  thresholds=thresholds)
    out["Acc24_metrics"]  = event_metrics_binary(acc_p,   acc_y,   acc_m,   thresholds=thresholds)

    return out

# ============================================================
# 5) Build data objects (same split as others)
# ============================================================

def build_data_objects(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15):
    prep = prepare_full_grid(df_rain)
    T = len(prep["times"])
    train_end, val_end = make_splits(T, train_frac=train_frac, val_frac=val_frac)

    X_scaled, scaler = scale_inputs_train_only(prep["X_raw"], train_end)

    thr3h = compute_thresholds_train(prep["Y_rain"], prep["M_y"], train_end, q=0.95)

    Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out=H_out)
    thrAcc24 = compute_thresholds_train(Acc24, MaskAcc24, train_end, q=0.95)

    ds_train = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=0, t_end=train_end,
        T_in=T_in, H_out=H_out
    )
    ds_val = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=train_end, t_end=val_end,
        T_in=T_in, H_out=H_out
    )
    ds_test = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=val_end, t_end=T,
        T_in=T_in, H_out=H_out
    )

    return prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), (train_end, val_end, T), (ds_train, ds_val, ds_test)


def make_loaders(ds_train, ds_val, ds_test, batch_size=32):
    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

# ============================================================
# 6) PERSISTENCE MODEL
# ============================================================

class PersistenceBaseline(nn.Module):
    """
    - Uses last observed rainfall at time t as prediction for all future H_out steps.
    - Deterministic quantiles: q10 = q50 = q90 = log1p(last_rain_mm).
    - Flash/Peak/Acc risks derived from these deterministic predictions.
    """
    def __init__(self,
                 num_stations,
                 T_in,
                 H_out,
                 quantiles,
                 rain_mean,
                 rain_std,
                 thr3h,
                 thrAcc24,
                 meteo_dim=7,
                 rain_idx_in_meteo=4):
        super().__init__()
        self.N = num_stations
        self.T_in = T_in
        self.H_out = H_out
        self.quantiles = list(quantiles)
        self.K = len(self.quantiles)

        self.meteo_dim = meteo_dim
        self.rain_idx_in_meteo = rain_idx_in_meteo

        self.rain_mean = torch.tensor(rain_mean, dtype=torch.float32, device=device)
        self.rain_std  = torch.tensor(rain_std,  dtype=torch.float32, device=device)

        self.thr3h    = torch.tensor(thr3h,    dtype=torch.float32, device=device)  # [N]
        self.thrAcc24 = torch.tensor(thrAcc24, dtype=torch.float32, device=device)  # [N]

    def forward(self, x, regime_id):
        """
        x: [B, T_in, N, F_all] where first meteo_dim dims are scaled meteo features
        regime_id: [B] (ignored for persistence)
        """
        B, T, N, F = x.shape
        assert T == self.T_in and N == self.N

        # extract scaled meteo part and pick rainfall dim
        x_meteo = x[..., :self.meteo_dim]        # [B,T,N,meteo_dim]
        rain_scaled = x_meteo[..., self.rain_idx_in_meteo]   # [B,T,N]
        rain_last_scaled = rain_scaled[:, -1, :]             # [B,N]

        # back to mm
        rain_last_mm = rain_last_scaled * self.rain_std + self.rain_mean  # [B,N]
        rain_last_mm = torch.clamp(rain_last_mm, min=0.0)

        # same for all H_out
        rain_hat_mm = rain_last_mm[:, None, :].expand(-1, self.H_out, -1)   # [B,H,N]

        # deterministic quantiles in log space
        q_log = torch.log1p(rain_hat_mm)   # [B,H,N]
        q_hat = q_log.unsqueeze(-1).expand(-1, -1, -1, self.K)  # [B,H,N,K]

        # --- Risk heads ---
        # Flash & Peak: check if last rainfall >= thr3h
        thr3h = self.thr3h[None, :]              # [1,N]
        flash_prob = (rain_last_mm >= thr3h).float()   # [B,N]
        peak_prob  = flash_prob.clone()               # same heuristic

        # Acc24: using accumulated predicted rainfall
        acc_sum = rain_hat_mm.sum(dim=1)              # [B,N]
        thrAcc = self.thrAcc24[None, :]
        acc_prob = (acc_sum >= thrAcc).float()        # [B,N]

        # convert probs -> logits
        eps = 1e-6
        def prob_to_logit(p):
            p = torch.clamp(p, eps, 1.0 - eps)
            return torch.log(p / (1.0 - p))

        flash_logits = prob_to_logit(flash_prob)   # [B,N]
        peak_logits  = prob_to_logit(peak_prob)    # [B,N]
        acc_logits   = prob_to_logit(acc_prob)     # [B,N]

        return q_hat, flash_logits, peak_logits, acc_logits

# ============================================================
# 7) Run persistence baseline on TEST split
# ============================================================

def run_persistence_baseline(df_rain,
                             T_in=16,
                             H_out=8,
                             train_frac=0.7,
                             val_frac=0.15,
                             batch_size=32,
                             quantiles=(0.1, 0.5, 0.9),
                             results_dir=RESULTS_SAVE_DIR):

    prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), splits, datasets = build_data_objects(
        df_rain,
        T_in=T_in,
        H_out=H_out,
        train_frac=train_frac,
        val_frac=val_frac,
    )
    train_end, val_end, T_total = splits
    ds_train, ds_val, ds_test = datasets
    _, _, test_loader = make_loaders(ds_train, ds_val, ds_test, batch_size=batch_size)

    N = len(prep["stations"])
    num_seasons = len(prep["unique_seasons"])
    print("Persistence baseline | N stations:", N, "| seasons:", num_seasons)

    # rainfall index in meteo features is 4, meteo_dim=7
    rain_idx = 4
    rain_mean = scaler.mean_[rain_idx]
    rain_std  = scaler.std_[rain_idx]

    model = PersistenceBaseline(
        num_stations=N,
        T_in=T_in,
        H_out=H_out,
        quantiles=quantiles,
        rain_mean=rain_mean,
        rain_std=rain_std,
        thr3h=thr3h,
        thrAcc24=thrAcc24,
        meteo_dim=7,
        rain_idx_in_meteo=rain_idx,
    ).to(device)

    scores_test = evaluate(model, test_loader, quantiles=quantiles)

    # save metrics
    out_path = os.path.join(results_dir, "persistence_test_metrics.json")
    import json
    with open(out_path, "w") as f:
        json.dump(scores_test, f, indent=2)

    print("\n[Persistence] Test metrics saved to:", out_path)
    print("CRPS_log:", scores_test["CRPS_log"], "| CRPS_mm:", scores_test["CRPS_mm"])
    print("Brier flash / peak / acc:",
          scores_test["Brier_flash"],
          scores_test["Brier_peak"],
          scores_test["Brier_acc"])

    return scores_test

persistence_results = run_persistence_baseline(
    df_rain=df_rain,
    T_in=16,
    H_out=8,
    train_frac=0.7,
    val_frac=0.15,
    batch_size=32,
    quantiles=(0.1, 0.5, 0.9),
    results_dir=RESULTS_SAVE_DIR,
)

print("\nPersistence baseline results summary:")
print(persistence_results)


# ---- Add CSV export here ----
import pandas as pd

flat = {
    "CRPS_log": persistence_results["CRPS_log"],
    "CRPS_mm": persistence_results["CRPS_mm"],
    "Brier_flash": persistence_results["Brier_flash"],
    "Brier_peak": persistence_results["Brier_peak"],
    "Brier_acc": persistence_results["Brier_acc"],
}

for i, (mae, rmse, n) in enumerate(
    zip(persistence_results["MAE_mm_by_lead"],
        persistence_results["RMSE_mm_by_lead"],
        persistence_results["n_valid_by_lead"])
):
    flat[f"MAE_lead{i+1}"] = mae
    flat[f"RMSE_lead{i+1}"] = rmse
    flat[f"n_valid_lead{i+1}"] = n

df = pd.DataFrame([flat])
csv_path = os.path.join(RESULTS_SAVE_DIR, "persistence_test_metrics.csv")
df.to_csv(csv_path, index=False)
print("Saved CSV to:", csv_path)




Using device: cpu
Results directory: C:\Users\pc\Documents\Rainfall\Baselines\Persistence\v1
Total stations: 34
Total unique times: 61360
Persistence baseline | N stations: 34 | seasons: 4


C:\Users\pc\AppData\Local\Temp\ipykernel_20024\605796846.py:325: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)



[Persistence] Test metrics saved to: C:\Users\pc\Documents\Rainfall\Baselines\Persistence\v1\persistence_test_metrics.json
CRPS_log: 0.21371083620698944 | CRPS_mm: 1.126069231255397
Brier flash / peak / acc: 0.05494580152502679 0.1773920620645585 0.06079429239275316

Persistence baseline results summary:
{'CRPS_log': 0.21371083620698944, 'CRPS_mm': 1.126069231255397, 'Brier_flash': 0.05494580152502679, 'Brier_peak': 0.1773920620645585, 'Brier_acc': 0.06079429239275316, 'MAE_mm_by_lead': [0.9338430166244507, 1.100321650505066, 1.1513938903808594, 1.1702393293380737, 1.1740361452102661, 1.1681021451950073, 1.1551722288131714, 1.1473643779754639], 'RMSE_mm_by_lead': [4.629355430603027, 5.140419960021973, 5.284617900848389, 5.344677448272705, 5.356433868408203, 5.342564105987549, 5.308084011077881, 5.285280227661133], 'n_valid_by_lead': [307503, 307500, 307497, 307494, 307491, 307488, 307485, 307482], 'Flash_metrics': {'n_valid': 307503, 'pr_auc': 0.16381599726603976, 'roc_auc': 0.6702573

# Climatology

In [ ]:
# ============================================================
# SCRIPT B: CLIMATOLOGY BASELINE (STATION–SEASON)
# ============================================================

import os, random, math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score

# ------------------------------------------------------------
# 0) Reproducibility + Device
# ------------------------------------------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ------------------------------------------------------------
# 0.5) Paths (EDIT)
# ------------------------------------------------------------
RESULTS_SAVE_DIR = r"C:\Users\pc\Documents\Rainfall\Baselines\Climatology\v1"
os.makedirs(RESULTS_SAVE_DIR, exist_ok=True)
print("Results directory:", RESULTS_SAVE_DIR)

# ============================================================
# 1) Utilities (same as before)
# ============================================================

class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.nanstd(X, axis=0)
        self.std_  = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_


def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()


def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    losses = []
    for k, q in enumerate(quantiles):
        losses.append(pinball_loss(y_hat[..., k], y_true, q, mask=mask))
    return 2.0 * torch.stack(losses).mean()


def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()


def safe_div(a, b, eps=1e-12):
    return a / (b + eps)


def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {
            "n_valid": int(valid.sum()),
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds},
        }

    p = probs[valid]
    y = y_true[valid]

    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try:
        roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except Exception:
        roc_auc = np.nan

    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP = ((yhat == 1) & (y == 1)).sum()
        FP = ((yhat == 1) & (y == 0)).sum()
        FN = ((yhat == 0) & (y == 1)).sum()

        POD = safe_div(TP, TP + FN)
        FAR = safe_div(FP, TP + FP)
        CSI = safe_div(TP, TP + FP + FN)

        by_thr[thr] = {"POD": float(POD), "FAR": float(FAR), "CSI": float(CSI)}

    return {
        "n_valid": int(valid.sum()),
        "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan,
        "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan,
        "by_thr": by_thr,
    }


def make_splits(T, train_frac=0.7, val_frac=0.15):
    train_end = int(T * train_frac)
    val_end   = int(T * (train_frac + val_frac))
    return train_end, val_end


def scale_inputs_train_only(X_raw, train_end):
    T, N, F = X_raw.shape
    X_flat = X_raw.reshape(T*N, F)
    train_flat = X_flat[:train_end*N]
    scaler = NaNIgnoringStandardScaler().fit(train_flat)
    X_scaled = scaler.transform(X_flat).reshape(T, N, F).astype(np.float32)
    return X_scaled, scaler


def compute_thresholds_train(Y, M, train_end, q=0.95):
    Y_train = Y[:train_end]
    M_train = M[:train_end]
    N = Y.shape[1]
    thr = np.zeros(N, dtype=np.float32)

    global_vals = Y_train[M_train > 0.5]
    global_fallback = np.nanpercentile(global_vals, q*100) if global_vals.size > 0 else 0.0

    for j in range(N):
        vals = Y_train[:, j][M_train[:, j] > 0.5]
        if len(vals) < 100:
            thr[j] = global_fallback
        else:
            thr[j] = np.nanpercentile(vals, q*100)
    return thr


def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc  = np.full((T, N), np.nan, dtype=np.float32)
    Mask = np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window = Y_rain[t+1:t+1+H_out]
        wmask  = M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok] = 1.0
        Acc[t, ok]  = window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing (same as before)
# ============================================================

def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)

    dr = df["DR"].astype(np.float32).to_numpy()
    dr_rad = np.deg2rad(dr)
    df["DR_sin"] = np.sin(dr_rad)
    df["DR_cos"] = np.cos(dr_rad)

    stations = (
        df[["StationID", "StationName", "Latitude", "Longitude"]]
        .drop_duplicates()
        .sort_values("StationID")
        .reset_index(drop=True)
    )
    stations["node_index"] = np.arange(len(stations))
    N = len(stations)
    print("Total stations:", N)

    times = np.sort(df["Datetime"].unique())
    T = len(times)
    print("Total unique times:", T)

    full_index = pd.MultiIndex.from_product(
        [times, stations["StationID"].values],
        names=["Datetime", "StationID"]
    )

    meteo_cols = [
        "DewPointTemperature",
        "StationLevelPressure",
        "SP",
        "Humidity",
        "Rainfall",
        "DR_sin",
        "DR_cos",
    ]

    df2 = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index()
    df_full = df2.reindex(full_index)
    X_raw = df_full.values.reshape(T, N, len(meteo_cols)).astype(np.float32)

    M_in = (~np.isnan(X_raw)).astype(np.float32)

    rain_idx = meteo_cols.index("Rainfall")
    Y_rain = X_raw[..., rain_idx]
    M_y    = M_in[..., rain_idx]

    dt_index = pd.DatetimeIndex(times)
    hour = dt_index.hour.astype(np.float32)
    month = dt_index.month.astype(np.float32)

    hour_sin = np.sin(2*np.pi*(hour/24.0))
    hour_cos = np.cos(2*np.pi*(hour/24.0))
    month_sin = np.sin(2*np.pi*((month-1)/12.0))
    month_cos = np.cos(2*np.pi*((month-1)/12.0))

    time_feats = np.stack([hour_sin, hour_cos, month_sin, month_cos], axis=-1).astype(np.float32)
    time_feats = np.repeat(time_feats[:, None, :], N, axis=1)

    season_by_time = (
        df.groupby("Datetime")["Season"]
          .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
          .reindex(times)
          .astype(str)
          .values
    )
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}
    season_ids = np.array([season_to_id[s] for s in season_by_time], dtype=np.int64)

    return {
        "stations": stations,
        "times": times,
        "X_raw": X_raw,
        "M_in": M_in,
        "Y_rain": Y_rain,
        "M_y": M_y,
        "meteo_cols": meteo_cols,
        "time_feats": time_feats,
        "season_ids": season_ids,
        "unique_seasons": unique_seasons,
    }

# ============================================================
# 3) Dataset (same as before)
# ============================================================

class BanglaRainDataset(Dataset):
    def __init__(
        self,
        X_scaled, M_in, time_feats,
        Y_rain, M_y,
        Acc24, MaskAcc24,
        season_ids,
        thr3h, thrAcc24,
        t_start, t_end,
        T_in=16, H_out=8,
        peak_min_obs=None,
    ):
        self.X_scaled = X_scaled
        self.M_in = M_in
        self.time_feats = time_feats
        self.Y_rain = Y_rain
        self.M_y = M_y
        self.Acc24 = Acc24
        self.MaskAcc24 = MaskAcc24
        self.season_ids = season_ids
        self.thr3h = thr3h
        self.thrAcc24 = thrAcc24
        self.T_in = T_in
        self.H_out = H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)

        self.indices = []
        for t in range(t_start + (T_in - 1), t_end - H_out):
            self.indices.append(t)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]

        x  = self.X_scaled[t-self.T_in+1:t+1]     # [T_in,N,F]
        m  = self.M_in[t-self.T_in+1:t+1]         # [T_in,N,F]
        tf = self.time_feats[t-self.T_in+1:t+1]   # [T_in,N,4]

        x = np.nan_to_num(x, nan=0.0).astype(np.float32)
        x_all = np.concatenate([x, m.astype(np.float32), tf], axis=-1)  # [T_in,N,F+F+4]

        y  = self.Y_rain[t+1:t+1+self.H_out]       # [H_out,N]
        my = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y, nan=0.0)).astype(np.float32)

        y_next = self.Y_rain[t+1]
        m_next = self.M_y[t+1].astype(np.float32)
        flash  = ((y_next >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mflash = m_next.copy()

        y_win = self.Y_rain[t+1:t+1+self.H_out]
        m_win = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)
        y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)

        acc  = self.Acc24[t]
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((acc >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)

        regime_id = int(self.season_ids[t])

        return (
            torch.from_numpy(x_all),
            torch.tensor(regime_id, dtype=torch.long),
            torch.from_numpy(y_log),
            torch.from_numpy(my),
            torch.from_numpy(flash), torch.from_numpy(mflash),
            torch.from_numpy(peak24), torch.from_numpy(mpeak),
            torch.from_numpy(acc24),  torch.from_numpy(macc),
        )

# ============================================================
# 4) Evaluation (same as other baselines)
# ============================================================

# ============================================================
# 4) Evaluation (same structure as other baselines)
# ============================================================

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.5)):
    model.eval()

    total_crps_log = 0.0
    total_crps_mm  = 0.0
    total_brier_flash = 0.0
    total_brier_peak  = 0.0
    total_brier_acc   = 0.0
    nb = 0

    H_out = None
    sum_abs = None
    sum_sq  = None
    count   = None

    qs = np.array(list(quantiles), dtype=np.float32)
    k50 = int(np.argmin(np.abs(qs - 0.5)))

    flash_p_list, flash_y_list, flash_m_list = [], [], []
    peak_p_list,  peak_y_list,  peak_m_list  = [], [], []
    acc_p_list,   acc_y_list,   acc_m_list   = [], [], []

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in loader:
        x = x.to(device)
        reg = reg.to(device)
        y_log = y_log.to(device)
        my = my.to(device)
        flash = flash.to(device); mflash = mflash.to(device)
        peak  = peak.to(device);  mpeak  = mpeak.to(device)
        acc   = acc.to(device);   macc   = macc.to(device)

        q_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        # CRPS in log space
        total_crps_log += crps_from_quantiles(q_hat, y_log, quantiles, mask=my).item()

        # Convert to mm and CRPS
        q_mm = torch.expm1(q_hat).clamp_min(0.0)
        y_mm = torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm  += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        # Risk Brier scores
        p_flash = torch.sigmoid(flash_logits)
        p_peak  = torch.sigmoid(peak_logits)
        p_acc   = torch.sigmoid(acc_logits)

        total_brier_flash += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak  += brier_score(p_peak,  peak,  mask=mpeak ).item()
        total_brier_acc   += brier_score(p_acc,   acc,   mask=macc  ).item()

        nb += 1

        # Median (0.5 quantile) for MAE/RMSE
        q50_log = q_hat[..., k50]
        q50_mm  = torch.expm1(q50_log).clamp_min(0.0)

        B, H, N = y_mm.shape
        if H_out is None:
            H_out = H
            sum_abs = torch.zeros(H_out, device="cpu")
            sum_sq  = torch.zeros(H_out, device="cpu")
            count   = torch.zeros(H_out, device="cpu")

        for h in range(H):
            m = (my[:, h, :] > 0.5)
            if m.any():
                err = (q50_mm[:, h, :][m] - y_mm[:, h, :][m]).detach().cpu()
                sum_abs[h] += err.abs().sum()
                sum_sq[h]  += (err**2).sum()
                count[h]   += m.sum().detach().cpu()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1))
        flash_y_list.append(flash.detach().cpu().reshape(-1))
        flash_m_list.append(mflash.detach().cpu().reshape(-1))

        peak_p_list.append(p_peak.detach().cpu().reshape(-1))
        peak_y_list.append(peak.detach().cpu().reshape(-1))
        peak_m_list.append(mpeak.detach().cpu().reshape(-1))

        acc_p_list.append(p_acc.detach().cpu().reshape(-1))
        acc_y_list.append(acc.detach().cpu().reshape(-1))
        acc_m_list.append(macc.detach().cpu().reshape(-1))

    out = {
        "CRPS_log":   total_crps_log / max(nb, 1),
        "CRPS_mm":    total_crps_mm  / max(nb, 1),
        "Brier_flash": total_brier_flash / max(nb, 1),
        "Brier_peak":  total_brier_peak  / max(nb, 1),
        "Brier_acc":   total_brier_acc   / max(nb, 1),
    }

    # MAE/RMSE by lead
    maes, rmses, counts = [], [], []
    for h in range(H_out):
        if count[h].item() < 1:
            maes.append(np.nan); rmses.append(np.nan); counts.append(0)
        else:
            maes.append(float((sum_abs[h] / count[h]).item()))
            rmses.append(float(torch.sqrt(sum_sq[h] / count[h]).item()))
            counts.append(int(count[h].item()))
    out.update({"MAE_mm_by_lead": maes, "RMSE_mm_by_lead": rmses, "n_valid_by_lead": counts})

    # Event metrics (Flash / Peak / Acc)
    flash_p = torch.cat(flash_p_list).numpy()
    flash_y = torch.cat(flash_y_list).numpy()
    flash_m = torch.cat(flash_m_list).numpy()

    peak_p  = torch.cat(peak_p_list).numpy()
    peak_y  = torch.cat(peak_y_list).numpy()
    peak_m  = torch.cat(peak_m_list).numpy()

    acc_p   = torch.cat(acc_p_list).numpy()
    acc_y   = torch.cat(acc_y_list).numpy()
    acc_m   = torch.cat(acc_m_list).numpy()

    out["Flash_metrics"]  = event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds)
    out["Peak24_metrics"] = event_metrics_binary(peak_p,  peak_y,  peak_m,  thresholds=thresholds)
    out["Acc24_metrics"]  = event_metrics_binary(acc_p,   acc_y,   acc_m,   thresholds=thresholds)

    return out


# ============================================================
# 5) Build data objects (same split as other baselines)
# ============================================================

def build_data_objects(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15):
    prep = prepare_full_grid(df_rain)
    T = len(prep["times"])
    train_end, val_end = make_splits(T, train_frac=train_frac, val_frac=val_frac)

    # We still scale inputs like other baselines (though climatology won't use them)
    X_scaled, scaler = scale_inputs_train_only(prep["X_raw"], train_end)

    # Thresholds for flash / acc24
    thr3h = compute_thresholds_train(prep["Y_rain"], prep["M_y"], train_end, q=0.95)

    Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out=H_out)
    thrAcc24 = compute_thresholds_train(Acc24, MaskAcc24, train_end, q=0.95)

    ds_train = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=0, t_end=train_end,
        T_in=T_in, H_out=H_out
    )
    ds_val = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=train_end, t_end=val_end,
        T_in=T_in, H_out=H_out
    )
    ds_test = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=val_end, t_end=T,
        T_in=T_in, H_out=H_out
    )

    return prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), (train_end, val_end, T), (ds_train, ds_val, ds_test)


def make_loaders(ds_train, ds_val, ds_test, batch_size=64):
    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


# ============================================================
# 6) Build station-wise climatology from TRAIN
# ============================================================

def build_climatology_from_train_loader(train_loader, num_stations, quantiles=(0.1, 0.5, 0.9)):
    """
    Build station-wise rainfall quantiles and risk probabilities
    from TRAIN data only.

    - Quantiles: based on distribution of 3-hourly rainfall (all leads combined).
    - Flash / Peak / Acc probabilities: station-wise frequency in TRAIN.
    """
    K = len(quantiles)

    # For rainfall quantiles (mm)
    values_per_station = [[] for _ in range(num_stations)]

    # For risk probabilities
    sum_flash = np.zeros(num_stations, dtype=np.float64)
    cnt_flash = np.zeros(num_stations, dtype=np.float64)
    sum_peak  = np.zeros(num_stations, dtype=np.float64)
    cnt_peak  = np.zeros(num_stations, dtype=np.float64)
    sum_acc   = np.zeros(num_stations, dtype=np.float64)
    cnt_acc   = np.zeros(num_stations, dtype=np.float64)

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in train_loader:
        # Move to CPU numpy
        y_mm = torch.expm1(y_log).clamp_min(0.0).cpu().numpy()  # [B,H,N]
        my_np = my.cpu().numpy()                                # [B,H,N]
        flash_np = flash.cpu().numpy()                          # [B,N]
        mflash_np = mflash.cpu().numpy()
        peak_np  = peak.cpu().numpy()
        mpeak_np = mpeak.cpu().numpy()
        acc_np   = acc.cpu().numpy()
        macc_np  = macc.cpu().numpy()

        B, H, N = y_mm.shape
        assert N == num_stations

        # Rainfall values per station
        for j in range(N):
            mask_j = my_np[:, :, j] > 0.5  # [B,H]
            if mask_j.any():
                vals = y_mm[:, :, j][mask_j]
                if vals.size > 0:
                    values_per_station[j].append(vals)

        # Risk labels per station
        sum_flash += (flash_np * mflash_np).sum(axis=0)
        cnt_flash += mflash_np.sum(axis=0)

        sum_peak  += (peak_np  * mpeak_np).sum(axis=0)
        cnt_peak  += mpeak_np.sum(axis=0)

        sum_acc   += (acc_np   * macc_np).sum(axis=0)
        cnt_acc   += macc_np.sum(axis=0)

    # Combine rainfall values and compute quantiles
    all_vals_global = []
    for j in range(num_stations):
        if len(values_per_station[j]) > 0:
            all_vals_global.append(np.concatenate(values_per_station[j], axis=0))
    if len(all_vals_global) > 0:
        all_vals_global = np.concatenate(all_vals_global, axis=0)
        global_q = np.nanpercentile(all_vals_global, np.array(quantiles)*100.0)
    else:
        global_q = np.zeros(K, dtype=np.float32)

    q_station_mm = np.zeros((num_stations, K), dtype=np.float32)
    for j in range(num_stations):
        if len(values_per_station[j]) == 0:
            q_station_mm[j, :] = global_q
        else:
            vals_j = np.concatenate(values_per_station[j], axis=0)
            if vals_j.size < 100:
                q_station_mm[j, :] = global_q
            else:
                q_station_mm[j, :] = np.nanpercentile(vals_j, np.array(quantiles)*100.0)

    # Station-wise risk probabilities
    eps = 1e-8
    p_flash = (sum_flash + eps) / (cnt_flash + 2*eps)
    p_peak  = (sum_peak  + eps) / (cnt_peak  + 2*eps)
    p_acc   = (sum_acc   + eps) / (cnt_acc   + 2*eps)

    # Convert quantiles to log space for compatibility with evaluate()
    q_station_log = np.log1p(q_station_mm).astype(np.float32)

    return q_station_log, p_flash.astype(np.float32), p_peak.astype(np.float32), p_acc.astype(np.float32)


# ============================================================
# 7) Climatology "model" (no training, just lookup)
# ============================================================

class ClimatologyBaseline(nn.Module):
    def __init__(self,
                 num_stations,
                 H_out,
                 quantiles,
                 q_station_log,  # [N,K] in log space
                 p_flash, p_peak, p_acc):
        super().__init__()
        self.N = num_stations
        self.H_out = H_out
        self.quantiles = list(quantiles)
        self.K = len(self.quantiles)

        # Register buffers so they move with .to(device)
        self.register_buffer("q_station_log", torch.from_numpy(q_station_log))  # [N,K]
        self.register_buffer("p_flash", torch.from_numpy(p_flash))              # [N]
        self.register_buffer("p_peak",  torch.from_numpy(p_peak))              # [N]
        self.register_buffer("p_acc",   torch.from_numpy(p_acc))               # [N]

    def forward(self, x, regime_id):
        """
        x: [B, T_in, N, F] (ignored, only used for shape)
        regime_id: [B] (ignored)
        """
        B, T, N, F = x.shape
        assert N == self.N

        # Quantile predictions in log space
        # q_station_log: [N,K] -> [1,1,N,K] -> [B,H_out,N,K]
        q = self.q_station_log[None, None, :, :].expand(B, self.H_out, N, self.K)

        # Station-wise probabilities -> logits
        eps = 1e-6
        def prob_to_logits(p):
            p = p.clamp(eps, 1.0 - eps)
            return torch.log(p / (1.0 - p))

        flash_logits_station = prob_to_logits(self.p_flash)  # [N]
        peak_logits_station  = prob_to_logits(self.p_peak)
        acc_logits_station   = prob_to_logits(self.p_acc)

        # Broadcast to batch: [B,N]
        flash_logits = flash_logits_station[None, :].expand(B, N)
        peak_logits  = peak_logits_station[None, :].expand(B, N)
        acc_logits   = acc_logits_station[None, :].expand(B, N)

        return q, flash_logits, peak_logits, acc_logits


# ============================================================
# 8) Run climatology baseline (VAL + TEST), save metrics
# ============================================================

# Hyperparameters for this baseline
T_IN   = 16
H_OUT  = 8
Q_LIST = (0.1, 0.5, 0.9)

# Build data & splits (SAME df_rain as other baselines)
prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), splits, datasets = build_data_objects(
    df_rain,
    T_in=T_IN,
    H_out=H_OUT,
    train_frac=0.7,
    val_frac=0.15,
)
train_end, val_end, T_total = splits
ds_train, ds_val, ds_test = datasets

train_loader, val_loader, test_loader = make_loaders(
    ds_train, ds_val, ds_test, batch_size=128
)

N_stations = len(prep["stations"])

# Build climatology from TRAIN only
q_station_log, p_flash, p_peak, p_acc = build_climatology_from_train_loader(
    train_loader,
    num_stations=N_stations,
    quantiles=Q_LIST,
)

# Create "model"
clim_model = ClimatologyBaseline(
    num_stations=N_stations,
    H_out=H_OUT,
    quantiles=Q_LIST,
    q_station_log=q_station_log,
    p_flash=p_flash,
    p_peak=p_peak,
    p_acc=p_acc,
).to(device)

print("\n[Climatology] Evaluating on validation set...")
val_scores = evaluate(clim_model, val_loader, quantiles=Q_LIST)
print("[Climatology] Evaluating on test set...")
test_scores = evaluate(clim_model, test_loader, quantiles=Q_LIST)

print("\n[Climatology] Validation metrics:")
for k, v in val_scores.items():
    if isinstance(v, (float, int)):
        print(f"  {k}: {v}")
print("\n[Climatology] Test metrics:")
for k, v in test_scores.items():
    if isinstance(v, (float, int)):
        print(f"  {k}: {v}")

# Save metrics
import json
val_path  = os.path.join(RESULTS_SAVE_DIR, "climatology_val_metrics.json")
test_path = os.path.join(RESULTS_SAVE_DIR, "climatology_test_metrics.json")
with open(val_path, "w") as f:
    json.dump(val_scores, f, indent=2)
with open(test_path, "w") as f:
    json.dump(test_scores, f, indent=2)


# ---- CSV export (flattened) ----
def flatten_scores_for_csv(name, scores):
    flat = {
        "Model": name,
        "CRPS_log": scores["CRPS_log"],
        "CRPS_mm": scores["CRPS_mm"],
        "Brier_flash": scores["Brier_flash"],
        "Brier_peak": scores["Brier_peak"],
        "Brier_acc": scores["Brier_acc"],
    }

    # Add lead-based metrics
    for i, (mae, rmse, n) in enumerate(
        zip(scores["MAE_mm_by_lead"],
            scores["RMSE_mm_by_lead"],
            scores["n_valid_by_lead"])
    ):
        flat[f"MAE_lead{i+1}"] = mae
        flat[f"RMSE_lead{i+1}"] = rmse
        flat[f"n_valid_lead{i+1}"] = n

    return flat


import pandas as pd

val_flat  = flatten_scores_for_csv("Climatology_Val",  val_scores)
test_flat = flatten_scores_for_csv("Climatology_Test", test_scores)

df = pd.DataFrame([val_flat, test_flat])
csv_path = os.path.join(RESULTS_SAVE_DIR, "climatology_metrics.csv")
df.to_csv(csv_path, index=False)

print("\n[Climatology] CSV saved ->", csv_path)


print("\n[Climatology] Saved:")
print("  Val metrics ->", val_path)
print("  Test metrics ->", test_path)


Using device: cpu
Results directory: C:\Users\pc\Documents\Rainfall\Baselines\Climatology\v1
Total stations: 34
Total unique times: 61360


C:\Users\pc\AppData\Local\Temp\ipykernel_20024\4155246182.py:323: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)



[Climatology] Evaluating on validation set...
[Climatology] Evaluating on test set...


C:\Users\pc\AppData\Local\Temp\ipykernel_20024\4155246182.py:323: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)



[Climatology] Validation metrics:
  CRPS_log: 0.16917014978308645
  CRPS_mm: 0.7928749420680106
  Brier_flash: 0.049775581664612725
  Brier_peak: 0.17619508748046225
  Brier_acc: 0.046576365650657356

[Climatology] Test metrics:
  CRPS_log: 0.14296056886410546
  CRPS_mm: 0.6711177551446276
  Brier_flash: 0.04164071084233001
  Brier_peak: 0.1552523972156147
  Brier_acc: 0.04039343046477168

[Climatology] CSV saved -> C:\Users\pc\Documents\Rainfall\Baselines\Climatology\v1\climatology_metrics.csv

[Climatology] Saved:
  Val metrics -> C:\Users\pc\Documents\Rainfall\Baselines\Climatology\v1\climatology_val_metrics.json
  Test metrics -> C:\Users\pc\Documents\Rainfall\Baselines\Climatology\v1\climatology_test_metrics.json
